In [1]:
# plot_loro_7b.py
# 运行方式: python plot_loro_7b.py

import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# ─── 路径配置 ───────────────────────────────────────────────
BASE = "/gemini/code"
OUTPUT_DIR = "/gemini/output/figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── 超参数 ────────────────────────────────────────────────
hyperparams = {
    "env": "FrozenLake-v1",
    "n_episodes": 150,
    "n_pretrain_eps": 30,
    "n_online_eps": 120,
    "n_exp": 5,
    "smooth": 10,
}

# ─── utils ────────────────────────────────────────────────
def smooth_curve(data, window):
    if window <= 1:
        return data
    result = np.copy(data)
    for i in range(len(data)):
        start = max(0, i - window // 2)
        end = min(len(data), i + window // 2 + 1)
        result[i] = np.mean(data[start:end])
    return result


def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)


# ─── 1. LLM baseline ───────────────────────────────────────
print("Loading Qwen dataset...")
path_7b = f"{BASE}/data/FrozenLake_Qwen2.5-7B-Instruct_Neps_200_20260428223255.pkl"
Qwen_7B_dataset = load_pkl(path_7b)

Qwen_7B_rewards = [
    Qwen_7B_dataset.episodes[i].compute_return()
    for i in range(hyperparams["n_pretrain_eps"])
]

Qwen_7B_mean = np.mean(Qwen_7B_rewards)
Qwen_7B_line = np.full(hyperparams["n_episodes"], Qwen_7B_mean)

print("Qwen7B mean:", Qwen_7B_mean)

# ─── 1.5 YOLO / SFT / DS baseline ─────────────────────────
YOLO_curves = {}

# SFT baseline（如果存在）
try:
    path_sft = f"{BASE}/data/FrozenLake_Qwen2.5-7B-Instruct_Neps_30_SFT.pkl"
    if os.path.exists(path_sft):
        sft_data = load_pkl(path_sft)
        sft_rewards = [
            ep.compute_return()
            for ep in sft_data.episodes[:hyperparams["n_pretrain_eps"]]
        ]
        YOLO_curves["SFT-7B"] = np.full(
            hyperparams["n_episodes"],
            np.mean(sft_rewards)
        )
        print("Loaded SFT baseline")
except:
    pass


# DS baseline（如果存在）
try:
    path_ds = f"{BASE}/data/FrozenLake_DeepSeek-7B.pkl"
    if os.path.exists(path_ds):
        ds_data = load_pkl(path_ds)
        ds_rewards = [
            ep.compute_return()
            for ep in ds_data.episodes[:hyperparams["n_pretrain_eps"]]
        ]
        YOLO_curves["DS-7B"] = np.full(
            hyperparams["n_episodes"],
            np.mean(ds_rewards)
        )
        print("Loaded DS baseline")
except:
    pass


# ─── 2. load caches ───────────────────────────────────────
cache_paths = {
    "cache30": [
        f"{BASE}/data/cache_FrozenLake_Neps_30.pkl",
        "/gemini/output/cache/cache_FrozenLake_Neps_30.pkl",
    ],
    "cache_rand": [
        f"{BASE}/data/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl",
        "/gemini/output/cache/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl",
    ],
}

loaded = {}
for k, paths in cache_paths.items():
    for p in paths:
        if os.path.exists(p):
            loaded[k] = load_pkl(p)
            print(f"Loaded {k}: {p}")
            break


# ─── 3. extract function（修复版）──────────────────────────
def extract_returns(cache, n_exp, n_eps, n_pre):
    r1000 = np.zeros((n_exp, n_eps))
    r3000 = np.zeros((n_exp, n_eps))

    n_online = n_eps - n_pre

    for i in range(n_exp):
        k1000 = f"pretrain_{n_pre}_eps_1000_steps_{i}"
        k3000 = f"pretrain_{n_pre}_eps_3000_steps_{i}"

        if k1000 in cache:
            v = cache[k1000]
            r1000[i, n_pre:] = v[:n_online]

        if k3000 in cache:
            v = cache[k3000]
            r3000[i, n_pre:] = v[:n_online]

    return r1000, r3000


# ─── 4. build curves ───────────────────────────────────────
curves = {}
n_eps = hyperparams["n_episodes"]
n_exp = hyperparams["n_exp"]
episodes_x = np.arange(n_eps)


# ─── LORO 30 eps ──────────────────────────────────────────
if "cache30" in loaded:
    r1000, r3000 = extract_returns(
        loaded["cache30"], n_exp, n_eps, 30
    )

    r1000[:, :30] = Qwen_7B_mean
    r3000[:, :30] = Qwen_7B_mean

    curves["LORO-1000 (pre=30)"] = (r1000.mean(0), r1000.std(0))
    curves["LORO-3000 (pre=30)"] = (r3000.mean(0), r3000.std(0))


# ─── RANDOM baseline（核心修复）────────────────────────────
if "cache_rand" in loaded:
    cache = loaded["cache_rand"]

    rand_1000 = np.zeros((n_exp, n_eps))
    rand_3000 = np.zeros((n_exp, n_eps))

    # 自动识别所有 eps（10/20/30）
    eps_list = [10, 20, 30]

    for i in range(n_exp):
        for eps in eps_list:

            k1 = f"pretrain_{eps}_eps_1000_steps_{i}_rand"
            k3 = f"pretrain_{eps}_eps_3000_steps_{i}_rand"

            if k1 in cache:
                v = cache[k1]
                L = min(len(v), n_eps)
                rand_1000[i, :L] = v[:L]

            if k3 in cache:
                v = cache[k3]
                L = min(len(v), n_eps)
                rand_3000[i, :L] = v[:L]

    curves["Random-1000"] = (rand_1000.mean(0), rand_1000.std(0))
    curves["Random-3000"] = (rand_3000.mean(0), rand_3000.std(0))


# ─── 5. plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    episodes_x,
    Qwen_7B_line,
    "--",
    color="gray",
    label="Qwen2.5-7B static",
)
# ─── YOLO lines ─────────────────────────────
yolo_colors = ["#9467bd", "#8c564b"]  # 紫 + 棕

for idx, (label, line) in enumerate(YOLO_curves.items()):
    ax.plot(
        episodes_x,
        line,
        linestyle=":",
        linewidth=2.2,
        color=yolo_colors[idx % len(yolo_colors)],
        label=label,
    )
    
colors = ["#d62728", "#2ca02c", "#1f77b4", "#ff7f0e"]

for idx, (label, (m, s)) in enumerate(curves.items()):
    m = smooth_curve(m, hyperparams["smooth"])
    s = smooth_curve(s, hyperparams["smooth"])

    ax.plot(episodes_x, m, color=colors[idx % len(colors)], label=label)
    ax.fill_between(episodes_x, m - s, m + s, alpha=0.2)

ax.axvline(30, linestyle="--", color="black")

ax.set_title("FrozenLake LORO + Random baseline")
ax.set_xlabel("Episodes")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

out_path = f"{OUTPUT_DIR}/FrozenLake_FINAL.png"
plt.savefig(out_path, dpi=200)
plt.close()

print("Saved:", out_path)

Loading Qwen dataset...


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen7B mean: 0.0
Loaded cache30: /gemini/code/data/cache_FrozenLake_Neps_30.pkl
Loaded cache_rand: /gemini/code/data/cache_FrozenLake_on_policy_pretrain_exp_rand.pkl
Saved: /gemini/output/figures/FrozenLake_FINAL.png
